In [3]:
import pandas as pd
import numpy as np
import re
import string
import joblib
from urllib.request import urlopen
import json
from rapidfuzz import process, fuzz

In [4]:
data_query=pd.read_csv(r'C:\projects\2nd_Semester_Project\data&models\intent_data.csv')
data_query_2=pd.read_csv(r"C:\projects\2nd_Semester_Project\data&models\full_noisy_levantine_dataset.csv")
full_data=pd.concat([data_query,data_query_2])
full_data=full_data.reset_index(drop=True)

In [5]:
def cleanText(text):
    table = str.maketrans('', '', 'ًٌٍَُِّْ')
    text = text.translate(table)
    text = text.replace('ى', 'ي').replace('ة', 'ه')
    text = re.sub(r'[أإآا]', 'ا', text)
    text = text.replace('ـ', '')
    text = re.sub(r'(\S)\1{2,}', r'\1', text)
    text = text.translate(str.maketrans('', '', string.punctuation + '؟،؛'))
    text = re.sub(r'[a-zA-Z]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text
mask=((full_data['Query'].str.len()>4) & (full_data['Query'].str.len()<100))
full_data=full_data[mask]


full_data['Query']=full_data['Query'].apply(cleanText)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [7]:
x=full_data.Query
y=full_data.Intent
x_train,x_test,y_train,y_test=train_test_split(x,y,train_size=0.8,stratify=y)

In [8]:
Tf_idf_vecorizer=TfidfVectorizer(ngram_range=(1,2))
x_train_Vectorized=Tf_idf_vecorizer.fit_transform(x_train)
x_test_Vectorized=Tf_idf_vecorizer.transform(x_test)

In [9]:
SVM_model=LinearSVC()
SVM_model.fit(x_train_Vectorized,y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [10]:
y_pred=SVM_model.predict(x_test_Vectorized)
print(classification_report(y_test,y_pred))

                           precision    recall  f1-score   support

          compare_drivers       0.99      1.00      1.00       344
get_constructor_standings       0.99      0.89      0.94       169
          get_driver_info       0.78      0.77      0.77       192
     get_driver_standings       0.70      0.76      0.73       186
        get_race_calendar       1.00      0.96      0.98       139
          get_race_result       0.99      1.00      1.00       210
     predict_next_n_races       1.00      1.00      1.00       494
        predict_next_race       0.99      1.00      0.99       216

                 accuracy                           0.94      1950
                macro avg       0.93      0.92      0.93      1950
             weighted avg       0.94      0.94      0.94      1950



In [11]:
joblib.dump(Tf_idf_vecorizer, r'C:\projects\2nd_Semester_Project\data&models\tfidf_vectorizer.pkl')

joblib.dump(SVM_model, r'C:\projects\2nd_Semester_Project\data&models\intent_classifier_svm.pkl')

['C:\\projects\\2nd_Semester_Project\\data&models\\intent_classifier_svm.pkl']

In [ ]:
class F1EntityExtractor:
    def __init__(self):
        self.vectorizer=joblib.load(r'C:\projects\2nd_Semester_Project\data&models\tfidf_vectorizer.pkl')
        self.svm_model=joblib.load(r"C:\projects\2nd_Semester_Project\data&models\intent_classifier_svm.pkl")
        self.Drivers_db = {
            # ريد بول (Red Bull)
            "ماكس فيرستابن": "max_verstappen",
            "ماكس فرستابن": "max_verstappen",
            "فيرستابن": "max_verstappen",
            "ماكس": "max_verstappen",
            "إيزاك هادجار": "isack_hadjar",
            "ايزاك هادجار": "isack_hadjar",

            # فيراري (Ferrari)
            "لويس هاميلتون": "lewis_hamilton",
            "لويس هاملتون": "lewis_hamilton",
            "هاميلتون": "lewis_hamilton",
            "لويس": "lewis_hamilton",
            "شارل لوكلير": "charles_leclerc",
            "شارل لوكليرك": "charles_leclerc",
            "لوكلير": "charles_leclerc",

            # مكلارين (McLaren)
            "لاندو نوريس": "lando_norris",
            "نوريس": "lando_norris",
            "لاندو": "lando_norris",
            "أوسكار بياستري": "oscar_piastri",
            "اوسكار بياستري": "oscar_piastri",
            "بياستري": "oscar_piastri",

            # مرسيدس (Mercedes)
            "جورج راسل": "george_russell",
            "راسل": "george_russell",
            "جورج": "george_russell",
            "كيمي أنتونيلي": "kimi_antonelli",
            "أنتونيلي": "kimi_antonelli",
            "اندريا كيمي انتونيلي": "kimi_antonelli",

            # أستون مارتن (Aston Martin)
            "فرناندو ألونسو": "fernando_alonso",
            "فرناندو الونسو": "fernando_alonso",
            "ألونسو": "fernando_alonso",
            "لانس سترول": "lance_stroll",
            "سترول": "lance_stroll",

            # ويليامز (Williams)
            "كارلوس ساينز": "carlos_sainz",
            "ساينز": "carlos_sainz",
            "أليكس ألبون": "alex_albon",
            "اليكس البون": "alex_albon",
            "ألبون": "alex_albon",

            # ألبين (Alpine)
            "بيير غاسلي": "pierre_gasly",
            "غاسلي": "pierre_gasly",
            "فرانكو كولابينتو": "franco_colapinto",
            "كولابينتو": "franco_colapinto",

            # هاس (Haas)
            "إستيبان أوكون": "esteban_ocon",
            "استيبان اوكون": "esteban_ocon",
            "أوكون": "esteban_ocon",
            "أوليفر بيرمان": "oliver_bearman",
            "اوليفر بيرمان": "oliver_bearman",
            "بيرمان": "oliver_bearman",

            # آر بي (Racing Bulls)
            "ليام لوسون": "liam_lawson",
            "لوسون": "liam_lawson",
            "أرفيد ليندبلاد": "arvid_lindblad",
            "ليندبلاد": "arvid_lindblad",

            # أودي (Audi/Sauber)
            "نيكو هولكنبرغ": "nico_hulkenberg",
            "هولكنبرغ": "nico_hulkenberg",
            "غابرييل بورتوليتو": "gabriel_bortoleto",
            "بورتوليتو": "gabriel_bortoleto",

            # كاديلاك (Cadillac)
            "سيرجيو بيريز": "sergio_perez",
            "سيرجيو بيرز": "sergio_perez",
            "بيريز": "sergio_perez",
            "تشيكو بيريز": "sergio_perez",
            "فالتيري بوتاس": "valtteri_bottas",
            "بوتاس": "valtteri_bottas",
        }

        self.Teams_db = {
            # Red Bull
            "ريدبول": "red_bull",
            "الريدبول": "red_bull",
            "ريد بول": "red_bull",
            
            # Ferrari
            "فيراري": "ferrari",
            "الفيراري": "ferrari",
            
            # Mercedes
            "ميرسيدس": "mercedes",
            "مرسيدس": "mercedes",
            "المرسيدس": "mercedes",
            
            # McLaren
            "مكلارين": "mclaren",
            "ماكلارين": "mclaren",
            "المكلارين": "mclaren",
            
            # Aston Martin
            "أستون مارتن": "aston_martin",
            "استون مارتن": "aston_martin",
            
            # Williams
            "ويليامز": "williams",
            "وليامز": "williams",
            
            # Alpine
            "ألبين": "alpine",
            "البين": "alpine",
            
            # Haas
            "هاس": "haas",
            
            # Racing Bulls (RB)
            "آر بي": "racing_bulls",
            "ار بي": "racing_bulls",
            "ريسينغ بولز": "racing_bulls",
            
            # Audi / Sauber
            "أودي": "audi",
            "اودي": "audi",
            "ساوبر": "audi",
            
            # Cadillac
            "كاديلاك": "cadillac",
        }

        self.Circuits_db = {
            "موناكو": "monaco",
            "سيلفرستون": "silverstone",
            "بريطانيا": "silverstone",
            "البحرين": "bahrain",
            "السعودية": "saudi_arabia",
            "جدة": "saudi_arabia",
            "أستراليا": "australia",
            "استراليا": "australia",
            "ملبورن": "australia",
            "اليابان": "japan",
            "سوزوكا": "japan",
            "الصين": "china",
            "شنغهاي": "china",
            "ميامي": "miami",
            "إيمولا": "imola",
            "ايمولا": "imola",
            "كندا": "canada",
            "مونتريال": "canada",
            "إسبانيا": "spain",
            "اسبانيا": "spain",
            "برشلونة": "spain",
            "النمسا": "austria",
            "المجر": "hungary",
            "بودابست": "hungary",
            "بلجيكا": "belgium",
            "سبا": "belgium",
            "هولندا": "netherlands",
            "زاندفورت": "netherlands",
            "إيطاليا": "italy",
            "ايطاليا": "italy",
            "مونزا": "italy",
            "أذربيجان": "azerbaijan",
            "اذربيجان": "azerbaijan",
            "باكو": "azerbaijan",
            "سنغافورة": "singapore",
            "أمريكا": "usa",
            "امريكا": "usa",
            "أوستن": "usa",
            "المكسيك": "mexico",
            "البرازيل": "brazil",
            "إنترلاغوس": "brazil",
            "لاس فيغاس": "las_vegas",
            "فيغاس": "las_vegas",
            "قطر": "qatar",
            "لوسيل": "qatar",
            "أبوظبي": "abu_dhabi",
            "أبو ظبي": "abu_dhabi",
            "ياس مارينا": "abu_dhabi",
        }

        self.Arabic_num = {
            "سباق": "1","واحد": "1",
            "اثنين": "2","اتنين": "2","سباقين": "2",
            "ثلاثة": "3","تلاثة": "3","اربعة": "4",'الاربع':"4",
            "أربعة": "4","خمسة": "5","الخمس":"5","ستة": "6","الست":"6",
            "سبعة": "7","ثمانية": "8","تمانية": "8",
            "تسعة": "9","عشرة": "10","عشر": "10"
        }

        self.Time_direction = {
            # Future
            "الجاي": "future","الجاية": "future",
            "بكرا": "future","بكرة": "future",
            "القادم": "future","القادمة": "future",
            
            # Past
            "الماضي": "past",
            "الماضية": "past","الفات": "past",
            "اللي فات": "past","السابق": "past",
            "السابقة": "past", "اليوم": "present",
            "الآن": "present", "الان": "present",
            "هلق": "present","هسا": "present"
        }
    
    def FuzzyMatching(self,Text):
        keys={"Circuits":self.Circuits_db,"Teams": self.Teams_db,"Drivers": self.Drivers_db}
        fixed_keys={"Time":self.Time_direction,"Number":self.Arabic_num}
        english_ID={}
    
        #match each word to find if its in time or number db and add them to the extarcted Text in english
        for word in Text.split():
            for name,db in fixed_keys.items():
                    if word in db:
                        english_ID[name]=db[word]
    
        #match the whole sentence using rapid fuzz technique which find the highest matched db so we can find the right command to trigger 
        for key,db in keys.items():
            results=process.extract(Text,db.keys(),scorer=fuzz.partial_ratio,limit=2)
            matched = {
            db[r[0]] for r in results if r[1] > 80
            }
            if matched:
                matched=list(matched)
                english_ID[key] = matched if len(matched) > 1 else matched[0]
        return english_ID
    
    def process_query(self,Text) :
        cleaned = cleanText(Text)
        
        # Classify intent
        vectorized = self.vectorizer.transform([cleaned])
        intent = self.svm_model.predict(vectorized)[0]
        
        # Extract entities from original Text (not cleaned — preserve numerals)
        entities = self.FuzzyMatching(Text)
        
        return {
            "intent": intent,
            "entities": entities,
            "raw_Text": Text
        }

In [15]:
if __name__=="__main__":
    extractor = F1EntityExtractor()
    test_query = "توقعلي الاربع السباقات الجاي"
    print(extractor.process_query(test_query))

{'intent': 'predict_next_race', 'entities': {'Number': '4', 'Time': 'future', 'Circuits': 'belgium'}, 'raw_Text': 'توقعلي الاربع السباقات الجاي'}
